# 05 - Aplicación Financiera Avanzada: Hierarchical Risk Parity (HRP) y Co-movimiento de Activos

## Curso de Machine Learning: Modelos No Supervisados

Durante más de 60 años, la **Teoría Moderna de Portafolios (Markowitz, 1952)** ha sido el pilar de la asignación cuantitativa de activos mediante la optimización media-varianza. Sin embargo, Markowitz adolece de una falla matemática crítica: **requiere invertir la matriz de covarianza ($\Sigma^{-1}$)**. Cuando los activos financieros están correlacionados (como ocurre en cualquier mercado real), el número de condición de $\Sigma$ explota, convirtiendo al optimizador en un "maximizador de errores de estimación".

En 2016, el Dr. **Marcos López de Prado** (Lawrence Berkeley National Laboratory / AQR Capital) propuso una de las aplicaciones más elegantes e influyentes del Machine Learning en finanzas cuantitativas: **Hierarchical Risk Parity (HRP)**.

HRP utiliza el **Clustering Jerárquico Aglomerativo** para organizar la matriz de correlación de los activos en un árbol de dependencias naturales y asigna pesos mediante **Bisección Recursiva Inversa a la Varianza**, logrando una diversificación óptima **sin invertir jamás una matriz de covarianza**.

---

### Contenidos
1. La Inestabilidad Numérica de Markowitz ($\Sigma^{-1}$) y la Falacia de la Descorrelación Plana
2. Paso 1 de HRP: Métrica Angular de Distancia de Correlación ($d_{i,j} = \sqrt{\frac{1}{2}(1 - \rho_{i,j})}$)
3. Paso 2 de HRP: Agrupamiento Jerárquico y Cuasi-Diagonalización de la Matriz de Covarianza
4. Paso 3 de HRP: Bisección Recursiva y Asignación de Pesos de Portafolio
5. Implementación Completa en Python y Simulación Histórica de 12 Activos
6. Benchmark de Desempeño: HRP vs. Portafolio Equi-Ponderado ($1/N$) vs. Inversa de Volatilidad
7. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, dendrogram

# Configuración visual
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. Paso 1: Métrica Angular de Distancia de Correlación

El coeficiente de correlación de Pearson $\rho_{ij} \in [-1, +1]$ **no es una métrica de distancia** válida porque no cumple la desigualdad triangular (ej. $1 - \rho$ tampoco es una métrica euclidiana).

López de Prado formaliza la **Distancia de Correlación Angular** entre dos series de retornos de activos $x_i$ y $x_j$:

$$d_{ij} = \sqrt{\frac{1}{2}(1 - \rho_{ij})} \in [0, 1]$$

- $\rho_{ij} = +1 \implies d_{ij} = 0$ (Activos perfectamente idénticos).
- $\rho_{ij} = 0 \implies d_{ij} = \frac{\sqrt{2}}{2} \approx 0.707$ (Activos independientes ortogonales).
- $\rho_{ij} = -1 \implies d_{ij} = 1$ (Activos perfectamente opuestos).


In [ ]:
# Simulación de 12 Activos Financieros pertenecientes a 3 Bloques Macro (Renta Variable, Renta Fija, Commodities)
activos = [
    'SP500', 'NASDAQ', 'EuroStoxx', 'Nikkei',         # Acciones Globales
    'US_10Y', 'Bund_10Y', 'UK_Gilt', 'TIPS_Inflation',# Bonos Soberanos
    'Oro', 'Plata', 'Petroleo_Brent', 'Cobre'        # Materias Primas
]

# Generamos 500 días de retornos con correlaciones de bloque realistas
rng = np.random.default_rng(42)
factores = rng.normal(0, 0.012, size=(500, 3))
retornos = np.column_stack([
    0.85 * factores[:, [0]] + rng.normal(0, 0.005, size=(500, 4)), # Bloque Acciones
    0.80 * factores[:, [1]] + rng.normal(0, 0.003, size=(500, 4)), # Bloque Bonos
    0.75 * factores[:, [2]] + rng.normal(0, 0.008, size=(500, 4))  # Bloque Commodities
])
df_retornos = pd.DataFrame(retornos, columns=activos)

# Matriz de correlación y matriz de covarianza histórica
matriz_corr = df_retornos.corr()
matriz_cov = df_retornos.cov()

# Matriz de distancias angulares d_ij
D_angular = np.sqrt(0.5 * (1.0 - matriz_corr))
np.fill_diagonal(D_angular.values, 0.0)


## 2. Paso 2: Agrupamiento Jerárquico y Cuasi-Diagonalización

Aplicamos clustering aglomerativo (enlace `single` o `average`) sobre la distancia angular. Al reordenar las filas y columnas de la matriz de covarianza según el orden de las hojas del dendrograma, **la matriz queda cuasi-diagonalizada**: los activos más correlacionados entre sí quedan situados lado a lado formando bloques densos a lo largo de la diagonal principal.


In [ ]:
# Construcción del enlace jerárquico a partir de la distancia angular
dist_vector = squareform(D_angular, checks=False)
linkage_hrp = linkage(dist_vector, method='single')

# Función de cuasi-diagonalización recursiva (reordenamiento según el dendrograma)
def obtener_orden_cuasi_diag(linkage_mat, n_activos):
    linkage_mat = linkage_mat.astype(int)
    orden = [linkage_mat[-1, 0], linkage_mat[-1, 1]]
    num_items = n_activos
    while max(orden) >= num_items:
        nuevo_orden = []
        for id_item in orden:
            if id_item >= num_items:
                idx_step = id_item - num_items
                nuevo_orden.append(linkage_mat[idx_step, 0])
                nuevo_orden.append(linkage_mat[idx_step, 1])
            else:
                nuevo_orden.append(id_item)
        orden = nuevo_orden
    return orden

orden_hrp = obtener_orden_cuasi_diag(linkage_hrp, len(activos))
activos_ordenados = [activos[i] for i in orden_hrp]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 1. Matriz de correlación original
sns.heatmap(matriz_corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, cbar=False, ax=axes[0])
axes[0].set_title('Matriz de Correlación Original (Desordenada)', fontsize=11, fontweight='bold')

# 2. Matriz de correlación cuasi-diagonalizada según el árbol
sns.heatmap(matriz_corr.loc[activos_ordenados, activos_ordenados], annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=axes[1])
axes[1].set_title('Matriz Cuasi-Diagonalizada por HRP (Bloques Naturales)', fontsize=11, fontweight='bold')

plt.suptitle('Cuasi-Diagonalización de Dependencias mediante Clustering Jerárquico', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 3. Paso 3: Bisección Recursiva y Asignación de Pesos

El árbol jerárquico se recorre de arriba hacia abajo (top-down). En cada división binaria entre dos sub-clusters $C_1$ y $C_2$:
1. Se calcula la **varianza del cluster** ponderada por paridad de riesgo inversa:
   $$\tilde{V}_k = w_k^T \Sigma_k w_k, \quad \text{donde } w_k = \frac{\text{diag}(\Sigma_k)^{-1}}{\mathbf{1}^T \text{diag}(\Sigma_k)^{-1} \mathbf{1}}$$
2. Se reparte el presupuesto de peso entre ambas ramas inversamente a sus varianzas:
   $$\alpha_1 = 1 - \frac{\tilde{V}_1}{\tilde{V}_1 + \tilde{V}_2}, \quad \alpha_2 = 1 - \alpha_1$$
3. El proceso continúa recursivamente hasta que cada activo recibe su peso final $w_i^*$.


In [ ]:
def calcular_varianza_cluster(cov_sub):
    inv_diag = 1.0 / np.diag(cov_sub)
    w = inv_diag / np.sum(inv_diag)
    return float(np.dot(np.dot(w, cov_sub), w))

def asignar_pesos_biseccion_recursiva(cov_ordenada, lista_items):
    pesos = pd.Series(1.0, index=lista_items)
    clusters = [lista_items]
    while len(clusters) > 0:
        nuevos_clusters = []
        for cl in clusters:
            if len(cl) > 1:
                split_idx = len(cl) // 2
                cl1 = cl[:split_idx]
                cl2 = cl[split_idx:]
                v1 = calcular_varianza_cluster(cov_ordenada.loc[cl1, cl1].values)
                v2 = calcular_varianza_cluster(cov_ordenada.loc[cl2, cl2].values)
                alpha = 1.0 - (v1 / (v1 + v2))
                pesos[cl1] *= alpha
                pesos[cl2] *= (1.0 - alpha)
                nuevos_clusters.extend([cl1, cl2])
        clusters = nuevos_clusters
    return pesos

cov_ordenada = matriz_cov.loc[activos_ordenados, activos_ordenados]
pesos_hrp = asignar_pesos_biseccion_recursiva(cov_ordenada, activos_ordenados)

# Modelos de referencia para comparación
pesos_equal = pd.Series(1.0 / len(activos), index=activos)
vol_inv = 1.0 / np.sqrt(np.diag(matriz_cov))
pesos_inv_vol = pd.Series(vol_inv / vol_inv.sum(), index=activos)

df_pesos = pd.DataFrame({
    'HRP_Jerarquico (%)': (pesos_hrp[activos] * 100).round(2),
    'Inversa_Volatilidad (%)': (pesos_inv_vol * 100).round(2),
    'Equi_Ponderado_1_N (%)': (pesos_equal * 100).round(2)
})

print('Pesos de Asignación por Activo según Cada Estrategia:')
display(df_pesos)

# Visualización de la asignación jerárquica
plt.figure(figsize=(12, 5))
df_pesos[['HRP_Jerarquico (%)', 'Inversa_Volatilidad (%)', 'Equi_Ponderado_1_N (%)']].plot(
    kind='bar', figsize=(12, 5), colormap='viridis', edgecolor='k', ax=plt.gca()
)
plt.title('Asignación de Portafolio: HRP Penaliza Bloques Altamente Correlacionados', fontsize=12, fontweight='bold')
plt.ylabel('Peso Asignado (%)', fontsize=11, fontweight='bold')
plt.xlabel('Activos del Portafolio', fontsize=11, fontweight='bold')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 4. Conclusiones

1. **Inmunidad Numérica**: HRP nunca invierte una matriz de covarianza, haciéndolo matemáticamente estable ante activos casi co-lineales donde Markowitz colapsa.
2. **Diversificación Estructural Auténtica**: Si un bloque (como las acciones tecnológicas) tiene 8 activos idénticos, Markowitz tiende a concentrar peso en ellos. HRP reconoce mediante el dendrograma que pertenecen a una sola rama y limita el presupuesto de riesgo de todo el bloque conjunto.

En el siguiente cuaderno, **06_Ejercicios_Practicos.ipynb**, pondrás en práctica todo el ciclo de análisis jerárquico aglomerativo sobre un caso real.
